# Tarea Guiada: Impacto del Escalado en la Frontera de Decisión

En esta tarea, intentarás alimentar un algoritmo k-NN con y sin aplicar `MinMaxScaler` usando un dataset con varianzas extremadamente dispares (ej. sueldo anual vs edad).

## 1. Carga y Simulación

In [4]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler

# Configurar semilla para reproducibilidad
np.random.seed(42)

# TODO: Crea un dataframe simulado con Edad (18-60) y Sueldo Anual (10k-200k) y un Label Binario de fraude[cite: 10, 11]
n_samples = 200

edad = np.random.randint(18, 61, size=n_samples)
sueldo_anual = np.random.randint(10000, 200001, size=n_samples)

# Generamos un label simulado de fraude basado en una regla combinada
fraude = np.where((edad < 30) | (sueldo_anual > 150000), 1, 0)

df = pd.DataFrame({
    'Edad': edad,
    'Sueldo_Anual': sueldo_anual,
    'Fraude': fraude
})

X = df[['Edad', 'Sueldo_Anual']]
y = df['Fraude']


## 2. Entrenamiento Crudo (Falla)

In [5]:
# TODO: Entrena un k-NN con el dataframe sin escalar y observa las predicciones[cite: 10, 12]
knn_crudo = KNeighborsClassifier(n_neighbors=5)
knn_crudo.fit(X, y)

# Probar predicción con dos usuarios que varían en Edad pero mantienen el mismo Sueldo[cite: 12]
usuario_1 = pd.DataFrame([[20, 50000]], columns=['Edad', 'Sueldo_Anual'])  # Joven
usuario_2 = pd.DataFrame([[55, 50000]], columns=['Edad', 'Sueldo_Anual'])  # Mayor

print("--- Predicción sin escalar (Datos Crudos) ---")
print(f"Usuario 1 (Edad 20, Sueldo 50k): Predicción = {knn_crudo.predict(usuario_1)[0]}")
print(f"Usuario 2 (Edad 55, Sueldo 50k): Predicción = {knn_crudo.predict(usuario_2)[0]}")


--- Predicción sin escalar (Datos Crudos) ---
Usuario 1 (Edad 20, Sueldo 50k): Predicción = 0
Usuario 2 (Edad 55, Sueldo 50k): Predicción = 0


## 3. Entrenamiento Limpio (MinMaxScaler)

In [6]:
# TODO: Aplica MinMaxScaler, entrena de nuevo y compara si el algoritmo ahora sí considera la Edad[cite: 10, 13]
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

knn_escalado = KNeighborsClassifier(n_neighbors=5)
knn_escalado.fit(X_scaled, y)

# Escalamos las muestras de prueba manteniendo el formato DataFrame para evitar warnings[cite: 13]
usuario_1_scaled = scaler.transform(usuario_1)
usuario_2_scaled = scaler.transform(usuario_2)

print("\n--- Predicción con MinMaxScaler ---")
print(f"Usuario 1 (Edad 20, Sueldo 50k): Predicción = {knn_escalado.predict(usuario_1_scaled)[0]}")
print(f"Usuario 2 (Edad 55, Sueldo 50k): Predicción = {knn_escalado.predict(usuario_2_scaled)[0]}")


--- Predicción con MinMaxScaler ---
Usuario 1 (Edad 20, Sueldo 50k): Predicción = 1
Usuario 2 (Edad 55, Sueldo 50k): Predicción = 0


## Evaluación Teórica / Decisión de Negocios

¿Por qué el algoritmo k-NN ignoró matemáticamente la Edad cuando entrenamos con los datos crudos?

R.-El algoritmo $k$-NN clasifica los puntos calculando la distancia entre ellos. La formula de la distancia euclidiana entre dos puntos $p=(x_1, y_1)$ y $q=(x_2, y_2)$ en dos dimensiones es:$$d(p, q) = \sqrt{(\text{Edad}_1 - \text{Edad}_2)^2 + (\text{Sueldo}_1 - \text{Sueldo}_2)^2}$$Efecto de la diferencia de magnitud:La variable Edad varía en un rango de $18$ a $60$ (diferencia máxima de $\approx 42$ unidades). Al elevar esta diferencia al cuadrado:$$(\Delta \text{Edad})^2 \leq 42^2 = 1\text{,}764$$La variable Sueldo Anual varía en un rango de $10\text{,}000$ a $200\text{,}000$ (diferencia máxima de $190\text{,}000$ unidades). Al elevar esta diferencia al cuadrado:$$(\Delta \text{Sueldo})^2 \leq (190\text{,}000)^2 = 36\text{,}100\text{,}000\text{,}000$$Conclusión matemática:
La magnitud del término $(\Delta \text{Sueldo})^2$ es hasta millones de veces mayor que $(\Delta \text{Edad})^2$. Como resultado, las diferencias de edad se vuelven irrelevantes dentro de la raíz cuadrada, haciendo que la distancia calculada dependa casi al $100\%$ del Sueldo Anual. Al aplicar MinMaxScaler, ambas variables se transforman al rango $[0, 1]$, permitiendo que tanto la Edad como el Sueldo_Anual tengan el mismo peso relativo en el cálculo de distancias.
